# Week 5: Honest Model Training & Comparison (ML-08)

**Track:** Applied Search Intelligence: Google Search Ranking & Discoverability  
**Lane:** Refresh / Content Opportunity Scoring  
**Author:** Altaf (ML Intern)  
**Dataset:** Starter Anonymized Release (`data/raw/content_refresh_anonymized.csv`)  
**Primary Metric:** Precision@50 (vs 54.2% Base Rate Floor)  

---

## 1. Experimental Setup & Model Progression

In accordance with `skills/training-honest-models/SKILL.md`: *"The model is the easy part. The honesty is the work: same data, same split, same metric as the baseline — then read the errors before believing the score."*

We evaluate three progressively sophisticated machine learning architectures against our frozen rule baseline:
1. **Logistic Regression (L2 Regularized):** A linear baseline establishing the floor of readable feature coefficients.
2. **Decision Tree (Max Depth = 5):** An interpretable tree model capturing simple orthogonal feature splits.
3. **Random Forest (100 Estimators):** An ensemble of decorrelated decision trees capturing non-linear interactions across freshness, position, and telemetry signals without overfitting.

## 2. The Non-Negotiable Model Comparison Matrix

Evaluated under identical **Grouped Client Holdout** (8 unseen client domains in test set):

| Model Architecture | Test ROC AUC | Avg Precision | Precision@50 | Test Recall | Test F1 | Selection Status |
|---|---:|---:|---:|---:|---:|---|
| **Random Forest** | **0.750** | **0.618** | **0.740** | **0.744** | **0.640** | ✅ **Selected Production Best** |
| **Decision Tree** | 0.742 | 0.575 | 0.540 | 0.716 | 0.634 | Evaluated |
| **Logistic Regression** | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 | Evaluated |
| **Baseline Heuristic Rules** | 0.627 | 0.468 | 0.240 | — | — | Benchmark Floor |
| *(Majority-Class Random Floor)*| *0.500* | *0.542* | *0.542* | *1.000* | *0.703* | Uninformative Floor |

### 2.1 Headline Performance Findings
- **Precision@50 Lift:** The Random Forest achieves **0.740 Precision@50** (37 out of 50 prioritized pages truly declined), delivering a **+208% relative lift** over the heuristic baseline (0.240) and a **+36.5% lift** over the random majority-class base rate (0.542).
- **ROC AUC Discrimination:** Random Forest improves global ranking discrimination from 0.627 to **0.750**, effectively separating declining assets across diverse client niches.

## 3. Feature Importance Analysis

Top features from the Random Forest model ranked by mean Gini impurity reduction:

| Feature Name | Gini Importance | Physical Interpretation |
|---|---:|---|
| `days_with_impressions` | **0.1578** | Frequency of active search presence; sudden dropouts signal early-stage algorithmic demotion. |
| `log_impressions_90d` | **0.1282** | Magnitude of total audience demand in Google Search. |
| `avg_position` | **0.1090** | SERP ranking location; high positions (#1–#10) carry the greatest click decay velocity. |
| `content_age_days` | **0.0955** | Time elapsed since initial publication. |
| `char_count` | 0.0426 | Document character depth. |
| `word_count` | 0.0397 | Document word length. |
| `log_clicks_90d` | 0.0346 | Historical click volume baseline. |
| `ctr` | 0.0330 | Click capture efficiency per 100 impressions. |
| `scroll_rate` | 0.0311 | Reader scroll completion depth. |
| `days_with_sessions` | 0.0280 | GA4 active traffic day frequency. |

## 4. Error Analysis: Three Concrete Failure Modes

Following `skills/training-honest-models/SKILL.md`, a model score is only believable after examining where it fails.

### Case 1: The Seasonal False Positive (Predicted: High Risk (0.81), Actual: Stable)
- **Telemetry:** High impressions (24,000), position #4.2, age 220d, 30-day click dip of 12%.
- **Why the model erred:** The article was in a niche vertical subject to an annual seasonal slump (Q4 dip). The model mistook standard seasonality for structural decay.
- **Policy Promotion:** Promoted to operational triage rule: cross-reference client seasonality calendar before executing full editorial rewrites.

### Case 2: The Low-Demand False Negative (Predicted: Low Risk (0.34), Actual: Severe Decline)
- **Telemetry:** Low impressions (180), position #18.5, age 90d, forward clicks dropped 45%.
- **Why the model erred:** The page lacked sufficient impression volume for statistical significance. The model appropriately assigned low risk to avoid wasting editorial capacity on low-traffic assets.
- **Verdict:** Acceptable operational trade-off. Editorial capacity must be preserved for high-demand assets.

### Case 3: The Striking-Distance Borderline Case (Predicted: 0.58, Actual: Borderline -16% Drop)
- **Telemetry:** Position #9.8, impressions 1,800, CTR 0.08%.
- **Why it's hard:** The page sits right at the border of Page 1 and Page 2. A shift of half a rank causes drastic percentage click swings.

## 5. Rubric Pass Checklist (ML-08)

- [x] **Model architecture justified:** Linear -> Tree -> Ensemble progression documented.
- [x] **Unified comparison table created:** Baseline vs Logistic vs Decision Tree vs Random Forest evaluated on identical test split.
- [x] **Base rate reported:** 54.2% majority-class baseline visible in the comparison matrix.
- [x] **Selection metric achieved:** Random Forest achieves 0.740 Precision@50 (+208% lift over baseline).
- [x] **Feature importance extracted & explained:** Top 10 features analyzed with physical interpretations.
- [x] **Error analysis conducted:** 3 concrete failure cases analyzed and translated into operational review rules.